In [1]:
"""
biblioteca_previsao_chuva.py

Biblioteca de apoio para o projeto de previsao de chuva (Aula 9).

O que esta biblioteca ESCONDE do aluno (nao e o foco pedagogico do dia):
    - construcao do pipeline de scikit-learn/imblearn
    - engenharia de features de tendencia (delta, media movel)
    - validacao cruzada estratificada sem vazamento de dados
    - busca do melhor threshold na curva precisao-recall

O que esta biblioteca MANTEM TRANSPARENTE (e o foco pedagogico do dia):
    - o NOME de cada tecnica de balanceamento disponivel, em TECNICAS_DISPONIVEIS
    - o NOME de cada modelo disponivel, em MODELOS_DISPONIVEIS
    - o aluno escolhe essas strings no script principal -- ele ve e decide
      QUAL tecnica esta sendo usada em cada estrategia, mesmo sem
      implementar o SMOTE ou o Random Forest por conta propria.

Esta versao imprime o progresso de cada etapa -- util em aula, para
que a turma veja que o codigo esta rodando (algumas estrategias,
como Random Forest com 5 folds, podem levar alguns minutos).
"""

import time

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, precision_recall_curve

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN


# ----------------------------------------------------------------------
# TECNICAS DE BALANCEAMENTO DISPONIVEIS -- visivel e editavel pelo aluno
# ----------------------------------------------------------------------
# Cada entrada e uma FUNCAO que constroi um amostrador novo (nao uma
# instancia pronta), para garantir que cada execucao comeca do zero.
TECNICAS_DISPONIVEIS = {
    None:            lambda: None,
    "undersampling": lambda: RandomUnderSampler(random_state=42),
    "smote":         lambda: SMOTE(random_state=42, k_neighbors=5),
    "smoteenn":      lambda: SMOTEENN(random_state=42),
}

# ----------------------------------------------------------------------
# MODELOS DISPONIVEIS -- visivel e editavel pelo aluno
# ----------------------------------------------------------------------
MODELOS_DISPONIVEIS = {
    "knn":    lambda: KNeighborsClassifier(n_neighbors=5),
    "logreg": lambda: LogisticRegression(class_weight="balanced", max_iter=1000),
    "rf":     lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", random_state=42),
    "rf_sem_peso": lambda: RandomForestClassifier(n_estimators=300, random_state=42),
}


def carregarDataset(caminhoCSV):
    """Carrega o CSV, faz a engenharia de features de tendencia e
    devolve (X, y) prontos para treinar. Detalhes de implementacao
    escondidos aqui de proposito -- o aluno recebe X e y prontos."""
    print(f"[1/4] Lendo {caminhoCSV}...")
    df = pd.read_csv(caminhoCSV)
    print(f"      {len(df)} linhas lidas.")

    print("[2/4] Normalizando a coluna alvo (choveu)...")
    df["choveu"] = df["choveu"].astype(str).str.lower().map(
        {"true": 1, "false": 0, "1": 1, "0": 0}
    )
    df = df.dropna(subset=["choveu"])
    df["choveu"] = df["choveu"].astype(int)
    df = df.sort_values("id").reset_index(drop=True)

    colunasBase = ["direcao_vento", "velocidade_vento", "umidade", "temperatura"]
    if "pressao" in df.columns:
        colunasBase.append("pressao")
        print("      Coluna 'pressao' encontrada e incluida nas features.")
    else:
        print("      AVISO: coluna 'pressao' nao encontrada neste CSV.")

    print("[3/4] Convertendo tipos e removendo linhas incompletas...")
    for coluna in colunasBase:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
    df = df.dropna(subset=colunasBase)

    print("[4/4] Calculando features de tendencia (delta e media movel)...")
    for coluna in colunasBase:
        df[f"delta_{coluna}"] = df[coluna].diff().fillna(0)
        df[f"media3_{coluna}"] = df[coluna].rolling(window=3, min_periods=1).mean()

    colunasFeatures = (
        colunasBase
        + [f"delta_{c}" for c in colunasBase]
        + [f"media3_{c}" for c in colunasBase]
    )
    df = df.dropna(subset=colunasFeatures + ["choveu"])

    print(f"      Pronto: {len(df)} registros, {len(colunasFeatures)} features.")
    print()

    return df[colunasFeatures], df["choveu"]


def avaliarEstrategia(nome, modeloChave, tecnicaChave, X, y, nFolds=5, silencioso=False):
    """Treina e avalia UMA estrategia (modelo + tecnica de balanceamento),
    usando validacao cruzada sem vazamento e threshold otimizado.
    modeloChave e tecnicaChave sao strings -- olhe MODELOS_DISPONIVEIS
    e TECNICAS_DISPONIVEIS para ver as opcoes."""
    if modeloChave not in MODELOS_DISPONIVEIS:
        raise ValueError(f"Modelo desconhecido: {modeloChave!r}. Opcoes: {list(MODELOS_DISPONIVEIS)}")
    if tecnicaChave not in TECNICAS_DISPONIVEIS:
        raise ValueError(f"Tecnica desconhecida: {tecnicaChave!r}. Opcoes: {list(TECNICAS_DISPONIVEIS)}")

    if not silencioso:
        print(f"  -> Rodando '{nome}' (modelo={modeloChave}, tecnica={tecnicaChave})...", end=" ", flush=True)
    inicio = time.time()

    modelo = MODELOS_DISPONIVEIS[modeloChave]()
    amostrador = TECNICAS_DISPONIVEIS[tecnicaChave]()

    etapas = [("escala", StandardScaler())]
    if amostrador is not None:
        etapas.append(("resample", amostrador))
    etapas.append(("modelo", modelo))
    pipeline = ImbPipeline(etapas)

    cv = StratifiedKFold(n_splits=nFolds, shuffle=True, random_state=42)
    probabilidades = cross_val_predict(pipeline, X, y, cv=cv, method="predict_proba")[:, 1]

    prAuc = average_precision_score(y, probabilidades)

    precisoes, recalls, thresholds = precision_recall_curve(y, probabilidades)
    f1s = np.divide(
        2 * precisoes * recalls, precisoes + recalls,
        out=np.zeros_like(precisoes), where=(precisoes + recalls) != 0,
    )
    melhorIdx = np.argmax(f1s[:-1]) if len(thresholds) > 0 else 0
    melhorThreshold = thresholds[melhorIdx] if len(thresholds) > 0 else 0.5

    resultado = {
        "nome": nome,
        "modelo": modeloChave,
        "tecnica": tecnicaChave,
        "pr_auc": prAuc,
        "threshold": melhorThreshold,
        "precisao": precisoes[melhorIdx],
        "recall": recalls[melhorIdx],
        "f1": f1s[melhorIdx],
    }

    tempoDecorrido = time.time() - inicio
    if not silencioso:
        print(f"concluido em {tempoDecorrido:.1f}s  |  PR-AUC={prAuc:.4f}  "
              f"precisao={resultado['precisao']:.2%}  recall={resultado['recall']:.2%}  F1={resultado['f1']:.3f}")

    return resultado


def compararEstrategias(estrategias, X, y):
    """Recebe uma lista de tuplas (nome, modeloChave, tecnicaChave),
    roda avaliarEstrategia para cada uma, e imprime o comparativo
    final ordenado por PR-AUC."""
    total = len(estrategias)
    print(f"Iniciando comparacao de {total} estrategias (5-fold cada)...")
    print("=" * 70)

    resultados = []
    inicioGeral = time.time()
    for indice, (nome, modeloChave, tecnicaChave) in enumerate(estrategias, start=1):
        print(f"[{indice}/{total}]", end=" ")
        resultados.append(avaliarEstrategia(nome, modeloChave, tecnicaChave, X, y))

    tempoTotal = time.time() - inicioGeral
    print("=" * 70)
    print(f"Todas as {total} estrategias concluidas em {tempoTotal:.1f}s.")
    print()

    resultados.sort(key=lambda r: r["pr_auc"], reverse=True)

    print("=" * 70)
    print("COMPARATIVO FINAL (ordenado por PR-AUC, do melhor para o pior)")
    print("=" * 70)
    print(f"{'Abordagem':<38} {'PR-AUC':>8} {'Precisao':>10} {'Recall':>8} {'F1':>7}")
    for r in resultados:
        print(f"{r['nome']:<38} {r['pr_auc']:>8.4f} {r['precisao']:>9.2%} {r['recall']:>7.2%} {r['f1']:>7.3f}")

    print()
    print(f"Taxa-base de chuva no dataset: {y.mean():.4%}")

    return resultados
